In [ ]:
using LensFactory
using LensFactory.Constants
using CairoMakie

# Initialize a cosmology

In [ ]:
# Initialize default cosmology
cosmo = Cosmology.init_cosmology()

# Lens and source redshifts
zl = 0.5
zs = 1.5

# ADDs and distance ratio
Dol = Cosmology.angular_diameter_distance(cosmo, 0., zl)
Dls = Cosmology.angular_diameter_distance(cosmo, zl, zs)
Dos = Cosmology.angular_diameter_distance(cosmo, 0., zs)
adis = Dls/Dos

# Create the grid
x, y = Lenses.get_meshgrid(5, 5, 0.065);

# Plummer Lens

In [ ]:
# Initialize a Plummer lens
lens = Lenses.init_PlummerLens(D_d=Dol,x_s=1.0, mass=1E12)

# Create an extended source with a Gaussian profile
src = Sources.gaussian(x, y,0.2, 0.2, (0.3, 0.3); A=1)

# Plot the magnification map in the image and source plane
fig, ax = Lenses.plot_image_plane(lens, x, y, adis; two_panel=true, figure_size=(800, 400), source=src)
display(fig)

# Plummer Lens + External Effects

In [ ]:
lens = Lenses.init_CompositeLens([(lens=:PlummerLens, D_d=Dol, x_c=0.0, y_c=0.0, x_s=1.0, mass=1E12),
                                  (lens=:ExternalEffects, kappa=0.0, gamma=0.5, angle=20)])

# Plot
fig, ax = Lenses.plot_image_plane(lens, x, y, adis; source=(0.1, 0.1))
display(fig)

# N-component SIS lens

In [ ]:
# To control the positions of point mass lenses
using Random
Random.seed!(1234)

# Initialize a lens made of multiple point mass lenses
n_point = 15
ensamble = [(lens=:PlummerLens, D_d=Dol, x_c=(-3.0 + 6.0*rand()), y_c = (-3.0 + 6.0*rand()), x_s=1.0, mass=2E11) for _ in 1:n_point]
lens = Lenses.init_CompositeLens(ensamble)

# Here we have split the source and image plane for better visualization using "two_panel" keyword
fig, ax = Lenses.plot_image_plane(lens, x, y, adis; two_panel=true, figure_size=(800, 400))
display(fig)


In [ ]:
fig, ax = Lenses.plot_surface_density(lens, x, y, adis; 
                                      D_d=Dol, 
                                      unit=:convergence, 
                                      heatmap_kws=(colorrange=(0.1, 3.0), colormap=:cubehelix), 
                                      plot_contour=true,
                                      contour_kws=(linewidth=1, levels=0.5:0.2:1.5, color=:white))
display(fig)

In [ ]:
fig, ax = Lenses.plot_magnification_map(lens, x, y, adis; 
                                        plane=:image, 
                                        heatmap_kws=(colorrange=(1,50), colormap=:viridis, colorscale=log10))
display(fig)

In [ ]:

fig, ax = Lenses.plot_magnification_map(lens, x, y, adis; 
                                        plane=:source, 
                                        rays_per_pixel=10, 
                                        heatmap_kws=(colorrange=(1,50), colormap=:viridis, colorscale=log10))
display(fig)

## 1D magnification profile (magnification vs area)

In [ ]:
fig, ax = Lenses.plot_magnification_profile(lens, x, y, adis; plane=:source, rays_per_pixel=10)
display(fig)